<a href="https://colab.research.google.com/github/KingGodFather007/Supervised-vs.-Unsupervised-ML/blob/main/supervised_vs_unsupervised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup — Auto-install Dependencies & Imports


In [ ]:
import importlib, subprocess, sys

def _ensure(package, import_name=None):
    name = import_name or package
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"Installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

_ensure("tensorflow")
_ensure("shap")
_ensure("lime")
_ensure("imbalanced-learn", "imblearn")
_ensure("xgboost")
_ensure("scikit-learn", "sklearn")
_ensure("pandas")
_ensure("numpy")
_ensure("matplotlib")
_ensure("seaborn")
_ensure("joblib")
_ensure("requests")
print("All dependencies ready.")

In [ ]:
import os, sys, time, warnings
import joblib, requests, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, precision_recall_curve,
    confusion_matrix, classification_report,
)
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

try:
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Input, Dense
    from tensorflow.keras.callbacks import EarlyStopping
    TF_OK = True
except ImportError:
    TF_OK = False
    print("TensorFlow not available — Autoencoder will be skipped.")

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False

warnings.filterwarnings("ignore")
print("Imports complete.")

## 2. Configuration


In [ ]:
BASE_DIR     = "experiment_results"
MODELS_DIR   = os.path.join(BASE_DIR, "models")
FIGURES_DIR  = os.path.join(BASE_DIR, "figures")
DATA_DIR     = "CICIDS2017_data"
RESULTS_CSV  = os.path.join(BASE_DIR, "model_comparison_results.csv")
PREPROCESSED = os.path.join(BASE_DIR, "preprocessed_data.joblib")
DATASET_URL  = "http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/MachineLearningCSV.zip"

for d in (BASE_DIR, MODELS_DIR, FIGURES_DIR):
    os.makedirs(d, exist_ok=True)
print("Directories ready.")

## 3. Data Loading & Preprocessing


In [ ]:
def download_and_load_data():
    zip_path = os.path.join(DATA_DIR, "MachineLearningCSV.zip")
    os.makedirs(DATA_DIR, exist_ok=True)
    if not os.path.exists(zip_path):
        print(f"Downloading dataset ...")
        r = requests.get(DATASET_URL, stream=True, timeout=600)
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
    try:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)
    except Exception as e:
        print(f"Extraction error: {e}"); return None
    frames = []
    for root, _, files in os.walk(DATA_DIR):
        for fname in files:
            if fname.endswith(".csv"):
                try:
                    df = pd.read_csv(os.path.join(root, fname),
                                     encoding="latin1", low_memory=False)
                    frames.append(df)
                    print(f"  Loaded {fname} {df.shape}")
                except Exception as e:
                    print(f"  Warning: {fname}: {e}")
    if not frames:
        return None
    merged = pd.concat(frames, ignore_index=True)
    print(f"Merged shape: {merged.shape}")
    return merged


def preprocess_data(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df.drop(columns=[c for c in ["Destination Port", "Fwd Header Length.1"]
                     if c in df.columns], inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    before = len(df)
    df.dropna(inplace=True); df.drop_duplicates(inplace=True)
    print(f"Rows removed: {before - len(df):,}  |  Remaining: {len(df):,}")
    df["Label"] = df["Label"].apply(
        lambda x: 0 if str(x).strip().upper() == "BENIGN" else 1)
    print("Class distribution:\n", df["Label"].value_counts(normalize=True))
    X = df.drop(columns=["Label"]); y = df["Label"]
    features = list(X.columns)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42)
    scaler = MinMaxScaler()
    X_tr_sc = pd.DataFrame(scaler.fit_transform(X_tr),
                            columns=features, index=X_tr.index)
    X_te_sc = pd.DataFrame(scaler.transform(X_te),
                            columns=features, index=X_te.index)
    rus = RandomUnderSampler(random_state=42)
    X_tr_r, y_tr_r = rus.fit_resample(X_tr_sc, y_tr)
    X_normal = X_tr_sc[y_tr == 0]
    print(f"Train (resampled): {X_tr_r.shape}  |  Test: {X_te_sc.shape}")
    return X_tr_r, X_te_sc, y_tr_r, y_te, features, X_normal


if os.path.exists(PREPROCESSED):
    print("Loading cached preprocessed data ...")
    X_train, X_test, y_train, y_test, features, X_normal = joblib.load(PREPROCESSED)
else:
    raw = download_and_load_data()
    X_train, X_test, y_train, y_test, features, X_normal = preprocess_data(raw)
    joblib.dump((X_train, X_test, y_train, y_test, features, X_normal), PREPROCESSED)

input_dim = X_train.shape[1]
print(f"\nFeatures: {input_dim}  |  Train: {X_train.shape}  |  Test: {X_test.shape}")

## 4. Model Training
### 4a. Supervised Models (GridSearchCV)


In [ ]:
RF_GRID  = {"n_estimators": [50, 100], "max_depth": [10, 20], "min_samples_split": [2, 5]}
XGB_GRID = {"n_estimators": [50, 100], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]}
IF_GRID  = {"n_estimators": [50, 100], "max_samples": ["auto", 0.5],
            "contamination": [0.1, 0.3, 0.5]}


def train_rf(X_tr, y_tr):
    print("Training Random Forest (GridSearchCV, 3-fold cv, f1_weighted) ...")
    t0 = time.time()
    gs = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                      RF_GRID, cv=3, scoring="f1_weighted", n_jobs=-1)
    gs.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    print(f"  Best params: {gs.best_params_}  ({elapsed:.0f}s)")
    return gs.best_estimator_, elapsed


def train_xgb(X_tr, y_tr):
    print("Training XGBoost (GridSearchCV, 3-fold cv, f1_weighted) ...")
    t0 = time.time()
    gs = GridSearchCV(
        XGBClassifier(use_label_encoder=False, eval_metric="logloss",
                      random_state=42, n_jobs=-1),
        XGB_GRID, cv=3, scoring="f1_weighted", n_jobs=-1)
    gs.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    print(f"  Best params: {gs.best_params_}  ({elapsed:.0f}s)")
    return gs.best_estimator_, elapsed


def train_if(X_tr, y_tr):
    # Grid-search Isolation Forest using training-set F1 as proxy.
    print("Training Isolation Forest (parameter grid search) ...")
    t0 = time.time()
    best_f1, best_clf = -1, None
    for n in IF_GRID["n_estimators"]:
        for s in IF_GRID["max_samples"]:
            for c in IF_GRID["contamination"]:
                clf = IsolationForest(n_estimators=n, max_samples=s,
                                      contamination=c, random_state=42, n_jobs=-1)
                clf.fit(X_tr)
                preds = np.where(clf.predict(X_tr) == -1, 1, 0)
                sc = f1_score(y_tr, preds, zero_division=0)
                if sc > best_f1:
                    best_f1, best_clf = sc, clf
    elapsed = time.time() - t0
    print(f"  Best contamination={best_clf.contamination}  ({elapsed:.0f}s)")
    return best_clf, elapsed


rf_path  = os.path.join(MODELS_DIR, "rf.joblib")
xgb_path = os.path.join(MODELS_DIR, "xgb.joblib")
if_path  = os.path.join(MODELS_DIR, "if.joblib")

rf,  rf_time  = (joblib.load(rf_path)  if os.path.exists(rf_path)
                 else train_rf(X_train, y_train))
xgb, xgb_time = (joblib.load(xgb_path) if os.path.exists(xgb_path)
                  else train_xgb(X_train, y_train))
ifo, if_time  = (joblib.load(if_path)  if os.path.exists(if_path)
                 else train_if(X_train, y_train))

if not os.path.exists(rf_path):  joblib.dump((rf,  rf_time),  rf_path)
if not os.path.exists(xgb_path): joblib.dump((xgb, xgb_time), xgb_path)
if not os.path.exists(if_path):  joblib.dump((ifo, if_time),  if_path)

print("Supervised + Isolation Forest models ready.")

### 4b. Autoencoder (Unsupervised)


In [ ]:
ae, ae_thresh, ae_time = None, None, 0.0

if TF_OK:
    ae_model_path  = os.path.join(MODELS_DIR, "autoencoder")
    ae_thresh_path = ae_model_path + ".threshold"

    if os.path.exists(ae_thresh_path):
        from tensorflow.keras.models import load_model
        print("Loading cached Autoencoder ...")
        ae = load_model(ae_model_path)
        ae_thresh = float(open(ae_thresh_path).read())
    else:
        print("Training Autoencoder on benign traffic only ...")
        t0 = time.time()
        inp = Input(shape=(input_dim,))
        x = Dense(64, activation="relu")(inp)
        x = Dense(16, activation="relu")(x)
        bn = Dense(8,  activation="relu")(x)
        x = Dense(16, activation="relu")(bn)
        x = Dense(64, activation="relu")(x)
        out = Dense(input_dim, activation="sigmoid")(x)
        ae = Model(inp, out)
        ae.compile(optimizer="adam", loss="mse")
        ae.summary()

        sample_size = min(len(X_normal), 200_000)
        X_samp = X_normal.sample(sample_size, random_state=42).values.astype(np.float32)
        es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        ae.fit(X_samp, X_samp, epochs=50, batch_size=512,
               validation_split=0.2, callbacks=[es], verbose=1)

        recon = ae.predict(X_samp, batch_size=512, verbose=0)
        errors = np.mean((X_samp - recon) ** 2, axis=1)
        ae_thresh = float(np.mean(errors) + 3 * np.std(errors))
        ae_time = time.time() - t0

        ae.save(ae_model_path)
        open(ae_thresh_path, "w").write(str(ae_thresh))
        print(f"AE threshold: {ae_thresh:.6f}  ({ae_time:.0f}s)")
else:
    print("Skipping Autoencoder — install TensorFlow to enable.")

## 5. Evaluation


In [ ]:
def compute_metrics(y_true, y_pred, y_score, name, train_time, infer_time):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0
    auc_val = roc_auc_score(y_true, y_score) if y_score is not None else float("nan")
    return {
        "Model": name,
        "Accuracy":   accuracy_score(y_true, y_pred),
        "Precision":  precision_score(y_true, y_pred, zero_division=0),
        "Recall":     recall_score(y_true, y_pred, zero_division=0),
        "F1-Score":   f1_score(y_true, y_pred, zero_division=0),
        "FPR": fpr, "FNR": fnr, "ROC-AUC": auc_val,
        "Train Time (s)":    round(train_time, 1),
        "Inference (s/1k)":  round(infer_time, 4),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    }


results = []

# Random Forest
t0 = time.time()
rf_pred  = rf.predict(X_test)
rf_score = rf.predict_proba(X_test)[:, 1]
results.append(compute_metrics(y_test, rf_pred, rf_score, "Random Forest",
                                rf_time, (time.time()-t0)/(len(y_test)/1000)))
print(classification_report(y_test, rf_pred, target_names=["BENIGN","ATTACK"]))

# XGBoost
t0 = time.time()
xgb_pred  = xgb.predict(X_test)
xgb_score = xgb.predict_proba(X_test)[:, 1]
results.append(compute_metrics(y_test, xgb_pred, xgb_score, "XGBoost",
                                xgb_time, (time.time()-t0)/(len(y_test)/1000)))
print(classification_report(y_test, xgb_pred, target_names=["BENIGN","ATTACK"]))

# Isolation Forest
t0 = time.time()
if_raw   = ifo.predict(X_test)
if_pred  = np.where(if_raw == -1, 1, 0)
if_score = -ifo.decision_function(X_test)
results.append(compute_metrics(y_test, if_pred, if_score, "Isolation Forest",
                                if_time, (time.time()-t0)/(len(y_test)/1000)))
print(classification_report(y_test, if_pred, target_names=["BENIGN","ATTACK"]))

# Autoencoder
if ae is not None:
    X_np = X_test.values.astype(np.float32)
    t0 = time.time()
    recon   = ae.predict(X_np, batch_size=512, verbose=0)
    ae_err  = np.mean((X_np - recon) ** 2, axis=1)
    ae_pred = (ae_err > ae_thresh).astype(int)
    results.append(compute_metrics(y_test, ae_pred, ae_err, "Autoencoder",
                                    ae_time, (time.time()-t0)/(len(y_test)/1000)))
    print(classification_report(y_test, ae_pred, target_names=["BENIGN","ATTACK"]))

results_df = pd.DataFrame(results).set_index("Model")
cols = ["Accuracy","Precision","Recall","F1-Score","FPR","FNR","ROC-AUC",
        "Train Time (s)","Inference (s/1k)"]
print("\n=== FINAL RESULTS TABLE ===")
print(results_df[[c for c in cols if c in results_df.columns]])
results_df.to_csv(RESULTS_CSV)
print(f"\nSaved to {RESULTS_CSV}")

## 6. Visualisation — ROC / PR Curves & Confusion Matrices


In [ ]:
def plot_roc_pr(y_true, y_score, name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Evaluation Curves — {name}", fontsize=13)
    fpr_v, tpr_v, _ = roc_curve(y_true, y_score)
    ax1.plot(fpr_v, tpr_v, lw=2, label=f"AUC = {auc(fpr_v, tpr_v):.4f}")
    ax1.plot([0, 1], [0, 1], "k--", lw=1)
    ax1.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
            title="ROC Curve", xlim=[0, 1], ylim=[0, 1.02])
    ax1.legend(); ax1.grid(alpha=0.3)
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    ax2.plot(rec, prec, lw=2, color="steelblue", label=f"AUC = {auc(rec, prec):.4f}")
    ax2.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve",
            xlim=[0, 1], ylim=[0, 1.02])
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{name.replace(' ', '_')}_roc_pr.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

def plot_cm(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["BENIGN", "ATTACK"],
                yticklabels=["BENIGN", "ATTACK"], ax=ax)
    ax.set(title=f"Confusion Matrix — {name}",
           xlabel="Predicted", ylabel="Actual")
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{name.replace(' ', '_')}_confusion.png"),
                dpi=150, bbox_inches="tight")
    plt.show()


plot_roc_pr(y_test, rf_score,  "Random Forest");    plot_cm(y_test, rf_pred,  "Random Forest")
plot_roc_pr(y_test, xgb_score, "XGBoost");          plot_cm(y_test, xgb_pred, "XGBoost")
plot_roc_pr(y_test, if_score,  "Isolation Forest"); plot_cm(y_test, if_pred,  "Isolation Forest")
if ae is not None:
    plot_roc_pr(y_test, ae_err, "Autoencoder");     plot_cm(y_test, ae_pred,  "Autoencoder")

# FPR / FNR summary bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, colour, title in [
    (axes[0], "FPR", "salmon",    "False Positive Rate (lower = better)"),
    (axes[1], "FNR", "steelblue", "False Negative Rate (lower = better)"),
]:
    results_df[col].plot(kind="bar", ax=ax, color=colour, edgecolor="k")
    ax.set_title(title, fontsize=11); ax.set_ylabel(col)
    ax.grid(axis="y", alpha=0.4); ax.tick_params(axis="x", rotation=30)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.3f}",
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "summary_fpr_fnr.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 7. SHAP Feature Importance


In [ ]:
if SHAP_OK:
    X_shap = X_test.sample(min(500, len(X_test)), random_state=42)
    for model, name in [(rf, "Random Forest"), (xgb, "XGBoost")]:
        print(f"\n--- SHAP: {name} ---")
        try:
            explainer   = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_shap)
            sv = shap_values[1] if isinstance(shap_values, list) else shap_values
            shap.summary_plot(sv, X_shap, show=False)
            plt.title(f"SHAP Feature Importance — {name}")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR,
                                     f"{name.replace(' ', '_')}_shap.png"),
                        dpi=150, bbox_inches="tight")
            plt.show()
        except Exception as e:
            print(f"SHAP error ({name}): {e}")
else:
    print("SHAP not installed — run _ensure('shap') to install.")

## Results Summary


In [ ]:
print("=" * 70)
print("TABLE 1 — Performance Metrics (test set)")
print("=" * 70)
cols = ["Accuracy", "Precision", "Recall", "F1-Score", "FPR", "FNR", "ROC-AUC"]
print(results_df[[c for c in cols if c in results_df.columns]])

print("\n" + "=" * 70)
print("TABLE 2 — Best Hyperparameters")
print("=" * 70)
for name, clf in [("Random Forest", rf), ("XGBoost", xgb), ("Isolation Forest", ifo)]:
    p = clf.get_params()
    keys = [k for k in p if k in
            ("n_estimators", "max_depth", "min_samples_split",
             "learning_rate", "contamination", "max_samples")]
    print(f"\n{name}:  " + "  |  ".join(f"{k}={p[k]}" for k in keys))

print(f"\nFigures saved to: {FIGURES_DIR}")
print(f"Results CSV:      {RESULTS_CSV}")